# Detection pipeline

**For real data:** edit `~/.piepy/config.json` so `paths.presentation` / `paths.analysis` point at your dirs. (Real sessions need their full artifacts — e.g. opto-pattern images for opto sessions — or parsing will raise.)

In [ ]:
import os
import polars as pl
import numpy as np
import natsort

from piepy.core.config import config
from piepy.core.registry import get_session_class
from piepy.core.hub import Hub, parse_session_name
from piepy.tasks.wheel_detection.wheelDetectionSession import WheelDetectionSession

from piepy.viz.plots import psychometric
from piepy.viz.trial.wheel_detection import trial_snapshot

pres = config.paths["presentation"][0]
print('presentation dir:', pres)

## Cohort across many sessions (`Hub`)
`Hub` runs each session in parallel and stacks them into one cohort table. It already isolates per-session failures (a bad session warns and is skipped).

In [ ]:
sessions = []
for t in os.listdir('/Users/kaan/data/training'):
    if "discrim" not in t and "DS_Store" not in t:
        res = parse_session_name(t)
        if res.animalid in ["KC153","KC154","KC155","KC156","KC158","KC159"]:
            sessions.append(t)
sessions = natsort.natsorted(sessions)
print(sessions)

In [ ]:
for s in sessions:
    try:
        sess = WheelDetectionSession(s)
        df = sess.analyze(load_flag=True)
    except Exception as e: 
        print(">>> WARNING <<< with session ")
        print(f"{e}")
        print("\n========\n")

In [ ]:
cohort = None
hub = Hub("wheel_detection")
hub.initialize([os.path.basename(s) for s in sessions], load_sessions=True)
cohort = hub.data
print('cohort:', None if cohort is None else cohort.shape)


In [ ]:
[c for c in cohort.columns if c.startswith("stat_")]

In [ ]:
import matplotlib.pyplot as plt
import behaviz as bv

STIM_COLORS = {"0.1cpd_4.0Hz":"#989898",
               "0.04cpd_8.0Hz":"#E08104",
               "0.16cpd_0.5Hz":"#2092D4"}

def session_progress(df, *, session_col="date", rt_outcome="hit", ax=None):
    """Per-session hit rate, early rate (left axis) and median reaction time (right axis).

    Args:
        df: a (multi-session) trial table.
        session_col: column identifying a session; sessions are ordered by it and numbered 1..N.
        rt_outcome: which trials the median reaction time is taken over (default the hit trials).
    """
    per = (
        df.group_by(session_col, maintain_order=True)
        .agg(
            (pl.col("stat_easy_hit_rate")).mean().alias("hit_rate"),
            (pl.col("stat_early_trial_count")/pl.col("stat_total_trial_count")).mean().alias("early_rate"),
            (pl.col("stat_easy_median_reaction_time")).mean().alias("median_rt"),
            (pl.col("stat_stim_trial_count")).mean().alias("trial_count"),
            (pl.col("stim_type").unique().drop_nulls()),
            (pl.col("level")).mean().alias("level")
        )
        .sort(session_col)
    ).with_row_index(name="session_nr",offset=1)  # session number

    if ax is None:
        _, ax = plt.subplots(figsize=(13, 6))
    f,ax = bv.plot_line(per["session_nr"], per["hit_rate"], marker="o",ax=ax,linewidth=2, color="#1B8A3A",label="Hit rate")
    f,ax = bv.plot_line(per["session_nr"], per["early_rate"] * 100, marker="o",ax=ax,linewidth=2, color="#F8381F",label="Early rate")

    stims = per["stim_type"].explode().unique(maintain_order=True).to_list()
    for stype in stims:
        levels = per.filter(pl.col("stim_type").list.contains(stype))["level"].unique()
        for lev in levels:
            lev_sessions = per.filter((pl.col("level")==lev) &
                                      (pl.col("stim_type").list.contains(stype)))["session_nr"].to_numpy().copy().astype(float)
            lev_sessions = np.insert(lev_sessions,0,lev_sessions[0]-0.1)
            lev_sessions = np.append(lev_sessions,lev_sessions[-1]+0.1)
            f,ax = bv.plot_line(x=lev_sessions,
                                y=[100+lev]*len(lev_sessions),
                                ax=ax, color=STIM_COLORS[stype],
                                linewidth=5,alpha=0.7)
            mid_point = (lev_sessions[0]+lev_sessions[-1])/2
            f,ax = bv.plot_text(mid_point,100+lev,f'{int(lev)}',ax=ax, color='#000000',ha="center",va="bottom")

    for i in per["session_nr"]:
        f,ax = bv.plot_text(i+0.05,
                            per["hit_rate"][i-1]+3,
                            f'{int(per["trial_count"][i-1])}',
                            ax=ax,
                            ha="center",
                            va="bottom")

    ax.set_xlabel("session")
    ax.set_ylabel("rate (%)")
    ax.set_ylim(0, 120)
    ax.set_yticks([0,25,50,75,100])

    ax2 = ax.twinx()  # second y-axis for reaction time
    f,ax2 = bv.plot_line(per["session_nr"], per["median_rt"], marker="s", ax=ax2,color="#0164E5")
    ax2.set_ylabel("median reaction time (ms)")
    ax2.grid(False)

    ax.spines["top"].set_visible(False)
    ax2.spines["top"].set_visible(False)
    ax2.spines["bottom"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax2.spines["left"].set_visible(False)

    ax.spines["left"].set_position(("outward", 8))
    ax2.spines["right"].set_position(("outward", 8))
    ax.spines["bottom"].set_position(("outward", 8))

    return per, (ax, ax2)

In [ ]:
animalids = ["KC153","KC154","KC155","KC156","KC158","KC159"]
f,axs = plt.subplots(len(animalids),1,figsize=(15,len(animalids)*5,))

for a,ax in zip(animalids,axs):
    data = cohort.filter(pl.col("animalid")==a)
    pr = session_progress(data,ax=ax)
    pr[1][0].set_title(a, fontsize=15)

### When looking at multiple sessions, the ```average_over``` argument is defaulted to ```animalid``` to average over animals, if ```None``` is passed all the trials will be pooled

In [ ]:
pr = hub.viz.psychometric(filterer={"stim_side":["catch","contra"]},
                          x="contrast",
                          outcome="outcome",
                          average_over="animalid",
                          compare="opto",
                          success="hit",
                          fit_curve=True,
                          model="weibull",
                          palette=("#DD7703","#0164E5"),
                          label='Opto ')

In [ ]:
pr = hub.viz.reaction_time_cloud(filterer={"outcome":"hit",
                                            "contrast":[0.125,0.5]},
                                  x="signed_contrast",
                                  value="reaction_time",
                                  compare="opto",
                                  average_over="animalid",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=50,
                                  label='Opto',
                                  dodge_width=0.3,
                                  violin_widths=0.2,
                                  violin_showextrema=False,
                                  violin_showmedians=True,
                                  width=0.1,
                                  scatter_s=50,
                                  scatter_linewidth=0.3,
                                  scatter_edgecolor="#FFFFFF")

In [ ]:
pr = hub.viz.reaction_time_dist(filterer={"outcome":"hit",
                                           "contrast":0.5},
                                  value="reaction_time",
                                  comparing="opto",
                                  average_over=None,
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=5,
                                  label='Opto ',
                                  alpha=0.8)